# DA6024: Meaning-Preserving Paraphrase Robustness

This notebook walks through the project and recreates every table and figure in the report from the saved outputs.

**No API keys are needed** for the cells that run by default. Steps that call Groq or Gemini are shown but switched off, because their outputs are already saved in `data/` and `results/`.

Run it from the project folder, using the project's virtual environment (see `README.md` for setup).

In [ ]:
import json, os, subprocess, sys
import pandas as pd
from IPython.display import Image, display

os.environ["HF_HUB_OFFLINE"] = os.environ.get("HF_HUB_OFFLINE", "0")
os.environ["PYTHONIOENCODING"] = "utf-8"
RUN_PAID_STEPS = False   # True reruns the Groq and Gemini steps (needs .env with API keys)
RUN_SLOW_STEPS = False   # True rebuilds levels, retrains models and rescores everything


def run(script, *args):
    print(">", script, *args)
    out = subprocess.run([sys.executable, os.path.join("src", script), *args], capture_output=True, text=True, encoding="utf-8")
    print(out.stdout[-3000:])
    if out.returncode:
        print(out.stderr[-3000:])

## 1. Data

The public datasets (PAWS, QQP, MRPC) are downloaded from Hugging Face. Our own data is already in `data/`: 300 base sentences, five levels of rewrites, the meaning review and the test pairs.

In [ ]:
if not os.path.exists("data/paws_test.csv"):
    run("download_data.py")
base = pd.read_csv("data/base_sentences.csv")
print(len(base), "base sentences;", base["source"].value_counts().to_dict())
base.head()

## 2. The five paraphrase levels

- **L1:** WordNet synonym swaps (`src/level1_synonyms.py`, locked)
- **L2 to L4:** Qwen with three prompts: structure only, minimal rewrite, aggressive rewrite (`src/levels_llm.py`, Groq key)
- **L5:** MarianMT, English to Chinese and back (`src/level5_marian.py`)

In [ ]:
if RUN_PAID_STEPS:
    run("levels_llm.py", "150")
if RUN_SLOW_STEPS:
    run("level5_marian.py", "150", "zh")
example = pd.read_csv("data/ladder.csv").query("label == 1 and id == 18")[["level", "s1", "s2"]]
example

## 3. Meaning check and test pairs

An NLI model scores each rewrite, the review decisions are applied, and each kept rewrite is paired with a different-meaning sentence (`src/make_pairs.py`).

In [ ]:
if RUN_SLOW_STEPS:
    run("meaning_check.py")
run("make_pairs.py")
pd.read_csv("results/meaning_summary.csv")

## 4. Train the classifiers, tune cutoffs and score every local method

These steps are slow on a laptop CPU, and their outputs are saved in `results/`. Set `RUN_SLOW_STEPS = True` to redo them.

In [ ]:
if RUN_SLOW_STEPS:
    run("tune_thresholds.py")
    run("run_tests.py")
    run("time_methods.py")
pd.DataFrame(json.load(open("results/thresholds.json"))).T[["cutoff", "val_accuracy"]]

## 5. LLM judge (Gemini)

`src/judge.py` saves every reply to `results/judge_gemini31.jsonl` and resumes where it stops. The free tier allows about 500 calls a day.

In [ ]:
if RUN_PAID_STEPS:
    run("judge.py", "gemini31")
print(sum(1 for _ in open("results/judge_gemini31.jsonl", encoding="utf-8")), "saved Gemini replies")

## 6. Results: tables and figures

In [ ]:
run("analyze.py")
pd.read_csv("results/ladder_by_level.csv").pivot(index="method", columns="level", values="accuracy").round(3)

In [ ]:
for fig in ["fig_degradation.png", "fig_recall_same.png", "fig_overlap.png", "fig_paws.png", "fig_cost.png"]:
    display(Image(os.path.join("results", fig), width=520))

In [ ]:
tests = pd.read_csv("results/test_sets.csv")
tests.pivot(index="method", columns="dataset", values="accuracy").round(3)

## 7. Extra checks: cutoffs, QQP overlap and the human check

In [ ]:
run("cutoff_test.py")
pd.read_csv("results/cutoff_test.csv").set_index("method")[["L4_original", "L4_no_paws", "L4_best"]].round(3)

In [ ]:
run("check_qqp_overlap.py")
run("human_check.py", "score")
run("human_check.py", "review")